<a href="https://colab.research.google.com/github/Sahilkom/Intern_project/blob/main/Task2/Modified_RAG_Hybrid_Search_RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Installing and Importing Required Dependencies

In [ ]:
!pip install langchain rank_bm25 pypdf unstructured chromadb
!pip install unstructured['pdf'] unstructured
!apt-get install poppler-utils
!apt-get install -y tesseract-ocr
!apt-get install -y libtesseract-dev
!pip install pytesseract
!pip install fpdf
!pip install langchain_community

In [2]:
from langchain.document_loaders import UnstructuredPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.vectorstores import Chroma

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

from langchain.embeddings import HuggingFaceInferenceAPIEmbeddings
from langchain.llms import HuggingFaceHub

from langchain.retrievers import BM25Retriever, EnsembleRetriever

from fpdf import FPDF

import os

# Pre-Processing Data

In [65]:
import json
def load_data():
    with open("/content/TASK_4.json", 'r') as f:
        return json.load(f)

data=load_data()

In [66]:
# create chunks
data_set = []
for table in data['tables']:
    table_info=[]
    table_name=f"Table name: {table['table_name']}"
    table_description=f"Description: {table['description']}"
    table_info.append(table_name)
    table_info.append(table_description)
    title="Column name   "
    table_info.append(title)
    index=1
    for col in table['columns']:
        col_info=f" {col['name']}"
        col_info=str(index)+". "+col_info
        table_info.append(col_info)
        index+=1
    if data_set.count(table_info) <= 0:
        data_set.append(table_info)


In [ ]:
for chunk in data_set:
    for info in chunk:
        print(info+"\n")
    print("\n")

In [71]:
pdf = FPDF()

for table in data_set:
  pdf.add_page()
  for info in table:
    pdf.set_font("Arial", size = 10)
    pdf.cell(2000, 10, txt = info, ln = 1, align = 'L')

pdf.output("new_data.pdf")

''

# Importing processed data in the form of Documnet

In [74]:
file_path = "/content/new_data.pdf"
data_file = UnstructuredPDFLoader(file_path)
docs = data_file.load()

In [ ]:
print(docs[0].page_content)

# Importing Feature Extraction Model

In [76]:
# Get Embedding Model from HF via API

from google.colab import userdata
HF_TOKEN = userdata.get('HUGGINGFACEHUB_API_TOKEN')

embeddings = HuggingFaceInferenceAPIEmbeddings(
    api_key=HF_TOKEN, model_name="BAAI/bge-base-en-v1.5"
)

### VectorStore

In [77]:
# Vector store with the selected embedding model
vectorstore = Chroma.from_documents(docs, embeddings)

In [78]:
# vectorstore_retreiver = vectorstore.as_retriever(search_kwargs={"k": 3})
vectorstore_retreiver = vectorstore.as_retriever()

In [79]:
keyword_retriever = BM25Retriever.from_documents(docs)
# keyword_retriever.k =  3

### Ensemble Retriever

In [80]:
ensemble_retriever = EnsembleRetriever(retrievers=[vectorstore_retreiver,
                                                   keyword_retriever],
                                       weights=[0.2,0.8])

In [81]:
llm = HuggingFaceHub(
    repo_id="mistralai/Mistral-7B-Instruct-v0.3",
    model_kwargs={"temperature": 0.3,"max_new_tokens":1024},
    huggingfacehub_api_token=HF_TOKEN,
)

### Prompt Template:

In [54]:
template = """
<|system|>>
                                              !! Hello !!
                                          This is AI Model-2.0
                                        How may I help you today?

CONTEXT: {context}
</s>
<|user|>
{query}
</s>
<|assistant|>
"""

In [55]:
prompt = ChatPromptTemplate.from_template(template)
output_parser = StrOutputParser()

In [56]:
chain = (
    {"context": ensemble_retriever, "query": RunnablePassthrough()}
    | prompt
    | llm
    | output_parser
)

# Queries

In [57]:
print(chain.invoke("Give me list of all tables present in data"))

Human: 
<|system|>>
                                              !! Hello !!
                                          This is AI Model-2.0
                                        How may I help you today?

CONTEXT: [Document(metadata={'source': '/content/new_data.pdf'}, page_content='Table name: products\n\nDescription: Stores product information\n\nColumn name\n\n1. product_id\n\n2. product_name\n\n3. category\n\n4. launch_date\n\n5. price\n\n6. manufacturer\n\n7. warranty_period\n\n8. stock_quantity\n\n9. rating\n\n10. dimensions\n\n11. weight\n\n12. color\n\n13. material\n\n14. power_usage\n\n15. model_number\n\nTable name: sales\n\nDescription: Stores sales information\n\nColumn name\n\n1. sale_id\n\n2. customer_id\n\n3. product_id\n\n4. sale_date\n\n5. quantity\n\n6. total_amount\n\n7. sales_channel\n\n8. payment_method\n\n9. discount\n\n10. sales_rep\n\n11. delivery_date\n\n12. delivery_status\n\n13. invoice_number\n\n14. shipping_cost\n\n15. tax_amount\n\nTable name: support_t

In [59]:
print(chain.invoke("give me tables that can be join on the basis of column"))

Human: 
<|system|>>
                                              !! Hello !!
                                          This is AI Model-2.0
                                        How may I help you today?

CONTEXT: [Document(metadata={'source': '/content/new_data.pdf'}, page_content='Table name: products\n\nDescription: Stores product information\n\nColumn name\n\n1. product_id\n\n2. product_name\n\n3. category\n\n4. launch_date\n\n5. price\n\n6. manufacturer\n\n7. warranty_period\n\n8. stock_quantity\n\n9. rating\n\n10. dimensions\n\n11. weight\n\n12. color\n\n13. material\n\n14. power_usage\n\n15. model_number\n\nTable name: sales\n\nDescription: Stores sales information\n\nColumn name\n\n1. sale_id\n\n2. customer_id\n\n3. product_id\n\n4. sale_date\n\n5. quantity\n\n6. total_amount\n\n7. sales_channel\n\n8. payment_method\n\n9. discount\n\n10. sales_rep\n\n11. delivery_date\n\n12. delivery_status\n\n13. invoice_number\n\n14. shipping_cost\n\n15. tax_amount\n\nTable name: support_t

In [60]:
print(chain.invoke("Give me all 10 table with their description"))

Human: 
<|system|>>
                                              !! Hello !!
                                          This is AI Model-2.0
                                        How may I help you today?

CONTEXT: [Document(metadata={'source': '/content/new_data.pdf'}, page_content='Table name: products\n\nDescription: Stores product information\n\nColumn name\n\n1. product_id\n\n2. product_name\n\n3. category\n\n4. launch_date\n\n5. price\n\n6. manufacturer\n\n7. warranty_period\n\n8. stock_quantity\n\n9. rating\n\n10. dimensions\n\n11. weight\n\n12. color\n\n13. material\n\n14. power_usage\n\n15. model_number\n\nTable name: sales\n\nDescription: Stores sales information\n\nColumn name\n\n1. sale_id\n\n2. customer_id\n\n3. product_id\n\n4. sale_date\n\n5. quantity\n\n6. total_amount\n\n7. sales_channel\n\n8. payment_method\n\n9. discount\n\n10. sales_rep\n\n11. delivery_date\n\n12. delivery_status\n\n13. invoice_number\n\n14. shipping_cost\n\n15. tax_amount\n\nTable name: support_t

In [61]:
print(chain.invoke("How many tables have a column related to price or cost?"))

Human: 
<|system|>>
                                              !! Hello !!
                                          This is AI Model-2.0
                                        How may I help you today?

CONTEXT: [Document(metadata={'source': '/content/new_data.pdf'}, page_content='Table name: products\n\nDescription: Stores product information\n\nColumn name\n\n1. product_id\n\n2. product_name\n\n3. category\n\n4. launch_date\n\n5. price\n\n6. manufacturer\n\n7. warranty_period\n\n8. stock_quantity\n\n9. rating\n\n10. dimensions\n\n11. weight\n\n12. color\n\n13. material\n\n14. power_usage\n\n15. model_number\n\nTable name: sales\n\nDescription: Stores sales information\n\nColumn name\n\n1. sale_id\n\n2. customer_id\n\n3. product_id\n\n4. sale_date\n\n5. quantity\n\n6. total_amount\n\n7. sales_channel\n\n8. payment_method\n\n9. discount\n\n10. sales_rep\n\n11. delivery_date\n\n12. delivery_status\n\n13. invoice_number\n\n14. shipping_cost\n\n15. tax_amount\n\nTable name: support_t

In [63]:
print(chain.invoke("Give me a primary key for each table"))

Human: 
<|system|>>
                                              !! Hello !!
                                          This is AI Model-2.0
                                        How may I help you today?

CONTEXT: [Document(metadata={'source': '/content/new_data.pdf'}, page_content='Table name: products\n\nDescription: Stores product information\n\nColumn name\n\n1. product_id\n\n2. product_name\n\n3. category\n\n4. launch_date\n\n5. price\n\n6. manufacturer\n\n7. warranty_period\n\n8. stock_quantity\n\n9. rating\n\n10. dimensions\n\n11. weight\n\n12. color\n\n13. material\n\n14. power_usage\n\n15. model_number\n\nTable name: sales\n\nDescription: Stores sales information\n\nColumn name\n\n1. sale_id\n\n2. customer_id\n\n3. product_id\n\n4. sale_date\n\n5. quantity\n\n6. total_amount\n\n7. sales_channel\n\n8. payment_method\n\n9. discount\n\n10. sales_rep\n\n11. delivery_date\n\n12. delivery_status\n\n13. invoice_number\n\n14. shipping_cost\n\n15. tax_amount\n\nTable name: support_t

In [64]:
print(chain.invoke("Give me foreign key relationship with department table and other table"))

Human: 
<|system|>>
                                              !! Hello !!
                                          This is AI Model-2.0
                                        How may I help you today?

CONTEXT: [Document(metadata={'source': '/content/new_data.pdf'}, page_content='Table name: products\n\nDescription: Stores product information\n\nColumn name\n\n1. product_id\n\n2. product_name\n\n3. category\n\n4. launch_date\n\n5. price\n\n6. manufacturer\n\n7. warranty_period\n\n8. stock_quantity\n\n9. rating\n\n10. dimensions\n\n11. weight\n\n12. color\n\n13. material\n\n14. power_usage\n\n15. model_number\n\nTable name: sales\n\nDescription: Stores sales information\n\nColumn name\n\n1. sale_id\n\n2. customer_id\n\n3. product_id\n\n4. sale_date\n\n5. quantity\n\n6. total_amount\n\n7. sales_channel\n\n8. payment_method\n\n9. discount\n\n10. sales_rep\n\n11. delivery_date\n\n12. delivery_status\n\n13. invoice_number\n\n14. shipping_cost\n\n15. tax_amount\n\nTable name: support_t